## Cell 1 — Setup and Environment

In [15]:
import os
import json
from groq import Groq
from IPython.display import display, HTML, JSON

os.environ["GROQ_API_KEY"] = "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"

client = client = None
MODEL = "meta-llama/llama-4-scout-17b-16e-instruct"

print("Setup complete.")
print(f"Model: {MODEL}")
print(f"API Key configured: {'Yes' if os.environ.get('GROQ_API_KEY') else 'No'}")

Setup complete.
Model: meta-llama/llama-4-scout-17b-16e-instruct
API Key configured: Yes


## Cell 2 — Synthetic Claims Data

Three structured provider cases based on real documented fraud patterns.
This represents the data a production system would retrieve from the payer's claims system, provider databases, and ML scoring output.

In [16]:
from synthetic_cases import CASES

print(f"Loaded {len(CASES)} synthetic cases:\n")
for case in CASES:
    print(f"  {case['case_id']}: {case['case_name']}")
    print(f"    Pattern: {case['fraud_type']}")
    print(f"    Source: {case['source_reference']}\n")

Loaded 3 synthetic cases:

  FWA-2026-001: Wound Allograft Overbilling
    Pattern: Billing for medically unnecessary services / diagnosis-procedure mismatch
    Source: Pattern consistent with DOJ June 2026 National Fraud Takedown - Wound Allograft Schemes

  FWA-2026-002: Complex E&M Upcoding - False Positive
    Pattern: Suspected upcoding - high complexity evaluation and management codes
    Source: Pattern consistent with Bluestone Physician Services $14.9M settlement - E&M upcoding

  FWA-2026-003: Behavioral Health Session Volume Anomaly
    Pattern: Billing for services not rendered / physically impossible session volume
    Source: Pattern consistent with DOJ June 2026 Takedown - behavioral health billing fraud ($49M Virginia scheme)



## Cell 3 — Investigation Agent System Prompt

The prompt that governs how the agent reasons through each case.
Key design decisions:
- Forces step-by-step chain reasoning
- Requires consideration of alternative explanations
- Enforces the compliance boundary: recommend, never decide
- Outputs structured JSON for auditability

In [17]:
from investigation_agent import INVESTIGATION_SYSTEM_PROMPT, build_investigation_prompt

print("Investigation Agent System Prompt:")
print("="*60)
print(INVESTIGATION_SYSTEM_PROMPT)

Investigation Agent System Prompt:
You are an expert healthcare payment integrity investigator AI assistant.
Your role is to autonomously investigate flagged claims and produce structured, 
investigation-ready case packages for SIU analysts.

You reason step by step through billing evidence, following this exact process:
1. Identify the core anomaly that triggered the flag
2. Evaluate whether the diagnosis codes clinically support the procedures billed
3. Assess billing volume against peer benchmarks and identify deviations
4. Check prior authorization compliance
5. Consider ALL alternative explanations that could legitimately explain the anomaly
6. Weigh the evidence for and against each explanation
7. Assign a confidence level: HIGH, MEDIUM, or LOW suspicion
8. Provide a clear recommended next action for the SIU analyst

CRITICAL RULES:
- You NEVER make denial decisions. You produce investigative arguments only.
- You ALWAYS consider alternative explanations before reaching a finding

## Cell 4 — Run Investigation Agent on All Three Cases

The agent intercepts each flagged claim and autonomously reasons through:
1. The core anomaly
2. Diagnosis-procedure clinical alignment
3. Billing volume versus peer benchmarks
4. Authorization compliance
5. Alternative explanations
6. Confidence assignment and recommendation

In [13]:
from investigation_agent import run_investigation_agent

agent_outputs = []

for case in CASES:
    output = run_investigation_agent(case)
    agent_outputs.append(output)

print("\nAll three cases investigated.")


INVESTIGATING: FWA-2026-001 - Wound Allograft Overbilling
Confidence Level: HIGH
Recommended Action: The SIU analyst should request additional documentation from Coastal Dermatology Associates, includi...

INVESTIGATING: FWA-2026-002 - Complex E&M Upcoding - False Positive
Confidence Level: LOW
Recommended Action: Review the claim in the context of the patient's medical record and the provider's billing patterns ...

INVESTIGATING: FWA-2026-003 - Behavioral Health Session Volume Anomaly
Confidence Level: HIGH
Recommended Action: The SIU analyst should conduct a detailed review of patient records, staff schedules, and billing do...

All three cases investigated.


## Cell 5 — Inspect Case 1: Wound Allograft Fraud (HIGH Suspicion)

Expected: Agent identifies diagnosis-procedure mismatch, volume anomaly, and missing authorization. Recommends escalation.

In [18]:
case1 = agent_outputs[0]

print(f"CASE: {case1['case_id']} — {case1['case_name']}")
print(f"\nCORE ANOMALY: {case1['core_anomaly']}")
print(f"\nCONFIDENCE LEVEL: {case1['confidence_level']}")
print(f"\nCONFIDENCE RATIONALE: {case1['confidence_rationale']}")
print(f"\nRECOMMENDED ACTION: {case1['recommended_action']}")
print(f"\nREASONING CHAIN:")
for step in case1.get('reasoning_chain', []):
    print(f"  Step {step['step']}: {step['check']}")
    print(f"    Finding: {step['finding']}")
    print(f"    Significance: {step['significance']}\n")

print("ALTERNATIVE EXPLANATIONS CONSIDERED:")
for alt in case1.get('alternative_explanations_considered', []):
    print(f"  Explanation: {alt['explanation']}")
    print(f"  Assessment: {alt['assessment']}\n")

CASE: FWA-2026-001 — Wound Allograft Overbilling

CORE ANOMALY: The claim for procedure code Q4121 (Amniotic wound allograft application) on 2026-05-15 for $14,500.00 with diagnosis code S00.01XA (Superficial abrasion of scalp, initial encounter) significantly exceeds peer averages and lacks prior authorization.

CONFIDENCE LEVEL: HIGH

CONFIDENCE RATIONALE: The combination of a significant billing deviation, lack of clinical support for the procedure, and absence of prior authorization strongly suggests potential fraud or abuse.

RECOMMENDED ACTION: The SIU analyst should request additional documentation from Coastal Dermatology Associates, including medical records supporting the necessity of the allograft application for the patient's condition, and investigate further to determine if there was a legitimate reason for the deviation from standard billing practices.

REASONING CHAIN:
  Step 1: Diagnosis code supports procedure
    Finding: False
    Significance: The diagnosis code S0

## Cell 6 — Inspect Case 2: E&M Upcoding False Positive (LOW Suspicion)

Expected: Agent finds the anomaly but correctly identifies the rural high-acuity patient population as a legitimate explanation. Recommends closing the flag.

In [19]:
case2 = agent_outputs[1]

print(f"CASE: {case2['case_id']} — {case2['case_name']}")
print(f"\nCORE ANOMALY: {case2['core_anomaly']}")
print(f"\nCONFIDENCE LEVEL: {case2['confidence_level']}")
print(f"\nCONFIDENCE RATIONALE: {case2['confidence_rationale']}")
print(f"\nRECOMMENDED ACTION: {case2['recommended_action']}")
print(f"\nREASONING CHAIN:")
for step in case2.get('reasoning_chain', []):
    print(f"  Step {step['step']}: {step['check']}")
    print(f"    Finding: {step['finding']}")
    print(f"    Significance: {step['significance']}\n")

print("ALTERNATIVE EXPLANATIONS CONSIDERED:")
for alt in case2.get('alternative_explanations_considered', []):
    print(f"  Explanation: {alt['explanation']}")
    print(f"  Assessment: {alt['assessment']}\n")

CASE: FWA-2026-002 — Complex E&M Upcoding - False Positive

CORE ANOMALY: The claim for procedure code 99205 (Office visit - new patient, high medical decision complexity) on 2026-05-20 for $320.00 was flagged for potential upcoding.

CONFIDENCE LEVEL: LOW

CONFIDENCE RATIONALE: The evidence suggests that the flagged claim may not be an instance of upcoding but rather a justified billing given the patient's high acuity and the provider's practice context.

RECOMMENDED ACTION: Review the claim in the context of the patient's medical record and the provider's billing patterns to confirm the legitimacy of the billing.

REASONING CHAIN:
  Step 1: Whether the diagnosis codes clinically support the procedures billed
    Finding: The diagnosis code E11.65 (Type 2 diabetes with hyperglycemia, with multiple comorbidities) supports the high complexity E&M coding given the patient's multiple chronic conditions.
    Significance: The patient's condition justifies the billed complexity level.

  St

## Cell 7 — Inspect Case 3: Behavioral Health Volume Anomaly (MEDIUM Suspicion)

This is the strongest demonstration of chain reasoning. The agent works through the arithmetic:
45 sessions ÷ 3 clinicians = 15 sessions each × 60 minutes = 900 minutes = 15 hours per clinician.
With documentation time: 15 × 75 minutes = 18.75 hours. Standard working day: 8 hours.
Physically impossible. Agent flags for targeted follow-up.

In [20]:
case3 = agent_outputs[2]

print(f"CASE: {case3['case_id']} — {case3['case_name']}")
print(f"\nCORE ANOMALY: {case3['core_anomaly']}")
print(f"\nCONFIDENCE LEVEL: {case3['confidence_level']}")
print(f"\nCONFIDENCE RATIONALE: {case3['confidence_rationale']}")
print(f"\nRECOMMENDED ACTION: {case3['recommended_action']}")
print(f"\nREASONING CHAIN:")
for step in case3.get('reasoning_chain', []):
    print(f"  Step {step['step']}: {step['check']}")
    print(f"    Finding: {step['finding']}")
    print(f"    Significance: {step['significance']}\n")

print("ALTERNATIVE EXPLANATIONS CONSIDERED:")
for alt in case3.get('alternative_explanations_considered', []):
    print(f"  Explanation: {alt['explanation']}")
    print(f"  Assessment: {alt['assessment']}\n")

CASE: FWA-2026-003 — Behavioral Health Session Volume Anomaly

CORE ANOMALY: The provider, ClearMind Behavioral Health Clinic, has billed an unusually high volume of individual psychotherapy sessions (90837) with a significant spike in sessions since January 15, 2026, exceeding peer averages by 275% and placing them in the 99th percentile among outpatient behavioral health providers in Richmond, VA.

CONFIDENCE LEVEL: HIGH

CONFIDENCE RATIONALE: The combination of a significant billing volume spike, feasibility concerns based on staffing levels, and a low rate of complete session documentation strongly suggests potential improper billing practices.

RECOMMENDED ACTION: The SIU analyst should conduct a detailed review of patient records, staff schedules, and billing documentation to verify the legitimacy of the billed services and assess the need for further investigation or potential site visit.

REASONING CHAIN:
  Step 1: Diagnosis code supports procedure
    Finding: The diagnosis co

## Cell 8 — LLM-as-Judge Quality Evaluation

Before any case package reaches the SIU analyst, an embedded evaluation layer quality-checks it across four dimensions:
1. Reasoning completeness
2. Evidence accuracy  
3. Step sequencing
4. Compliance adherence

This addresses the behavioral failure risk documented in Akshathala et al. (2026) and produces the native audit trail AMA June 2026 policy demands.

In [21]:
from investigation_agent import run_judge_evaluation

judge_outputs = []

for i, case in enumerate(CASES):
    judge_result = run_judge_evaluation(case, agent_outputs[i])
    judge_outputs.append(judge_result)

print("\nJudge Evaluation Complete.")
print("\nSUMMARY:")
for i, judge in enumerate(judge_outputs):
    print(f"  {CASES[i]['case_id']}: Overall {judge.get('overall_score','N/A')}/5 | Passes QA: {judge.get('passes_quality_threshold','N/A')}")


Running Judge Evaluation for FWA-2026-001...
Judge Overall Score: 5/5
Passes Quality Threshold: True

Running Judge Evaluation for FWA-2026-002...
Judge Overall Score: 5/5
Passes Quality Threshold: True

Running Judge Evaluation for FWA-2026-003...
Judge Overall Score: 5/5
Passes Quality Threshold: True

Judge Evaluation Complete.

SUMMARY:
  FWA-2026-001: Overall 5/5 | Passes QA: True
  FWA-2026-002: Overall 5/5 | Passes QA: True
  FWA-2026-003: Overall 5/5 | Passes QA: True


## Cell 9 — Generate HTML Report

The notebook generates a clean HTML file rendering all three case packages.
This is what the SIU analyst would see in a production interface.

In [22]:
from report_generator import generate_html_report

results = [
    {"case": CASES[i], "agent_output": agent_outputs[i], "judge_output": judge_outputs[i]}
    for i in range(len(CASES))
]

html_report = generate_html_report(results)

with open("fwa_investigation_report.html", "w") as f:
    f.write(html_report)

with open("results.json", "w") as f:
    json.dump(results, f, indent=2)

print("HTML report saved: fwa_investigation_report.html")
print("Raw JSON results saved: results.json")
print("\nOpen fwa_investigation_report.html in your browser to view the case packages.")

HTML report saved: fwa_investigation_report.html
Raw JSON results saved: results.json

Open fwa_investigation_report.html in your browser to view the case packages.
